In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os, glob, lmdb, pickle, argparse
import numpy as np

def as_energy(obj):
    """
    从样本对象里稳健地取出能量数值（float）。
    兼容 attribute / dict 两种风格，以及 torch/tensor -> float。
    """
    val = None
    # 优先 energy，再退回 y
    for key in ("energy", "y"):
        # 先试 attribute
        if hasattr(obj, key):
            val = getattr(obj, key)
            break
        # 再试 dict-like
        if hasattr(obj, "get"):
            v = obj.get(key, None)
            if v is not None:
                val = v
                break
    if val is None:
        raise KeyError("sample does not contain energy or y field")
    try:
        # 兼容 numpy / torch tensor
        return float(val) if np.isscalar(val) else float(np.array(val).reshape(()))
    except Exception:
        # 最后兜底
        return float(val)

def as_fid(obj):
    """
    取出 fid（int）。作者制作 LMDB 时，最优样本 fid = -1。
    """
    if hasattr(obj, "fid"):
        return int(getattr(obj, "fid"))
    if hasattr(obj, "get"):
        v = obj.get("fid", None)
        if v is not None:
            return int(v)
    raise KeyError("sample does not contain fid field")

def read_length(txn):
    """
    读取分片里的样本总数：作者写入了 'length'。
    若没有，就线性探测（不推荐，但做兜底）。
    """
    raw = txn.get(b"length")
    if raw is not None:
        try:
            return int(pickle.loads(raw))
        except Exception:
            pass
    # 兜底线性探测（慢）：从 0 开始直到 miss 连续 N 次（N=100）
    print("WARNING: 'length' key not found, falling back to slow probing...")
    miss = 0
    i = 0
    while miss < 100:  # 简单兜底
        if txn.get(str(i).encode("ascii")) is None:
            miss += 1
        else:
            miss = 0
        i += 1
    return max(0, i - miss)

def scan_lmdb(db_path, atol):
    total = 0
    n_fid_neg1 = 0
    bad = []  # 存储 (db_path, key_str, energy) 违反条件的样本
    with lmdb.open(db_path, subdir=False, readonly=True, lock=False,
                   readahead=True, meminit=False, max_readers=1) as env:
        with env.begin() as txn:
            length = read_length(txn)
            total = length
            for i in range(length):
                key = str(i).encode("ascii")
                raw = txn.get(key)
                if raw is None:
                    # 有些分片可能存在空洞，跳过
                    continue
                obj = pickle.loads(raw)
                try:
                    fid = as_fid(obj)
                except KeyError:
                    # 没有 fid 的样本跳过
                    continue
                if fid == -1:
                    n_fid_neg1 += 1
                    try:
                        e = as_energy(obj)
                    except KeyError:
                        bad.append((db_path, key.decode(), "NO_ENERGY_FIELD"))
                        continue
                    if not np.isclose(e, 0.0, atol=atol):
                        bad.append((db_path, key.decode(), e))
    return total, n_fid_neg1, bad

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--lmdb_dir", default="train_allE", help="包含若干 *.lmdb 分片的目录")
    ap.add_argument("--pattern", default="*.lmdb", help="分片文件匹配模式，默认 *.lmdb")
    ap.add_argument("--atol", type=float, default=1e-8, help="判断能量等于 0 的绝对容差")
    args = ap.parse_args()

    lmdb_files = sorted(glob.glob(os.path.join(args.lmdb_dir, args.pattern)))
    if not lmdb_files:
        print(f"[ERROR] 没在 {args.lmdb_dir} 下找到匹配 {args.pattern} 的 lmdb 文件")
        return

    grand_total = 0
    grand_fid_neg1 = 0
    all_bad = []

    print(f"扫描 {len(lmdb_files)} 个分片...\n")
    for path in lmdb_files:
        t, nneg1, bad = scan_lmdb(path, args.atol)
        grand_total += t
        grand_fid_neg1 += nneg1
        all_bad.extend(bad)
        print(f"- {os.path.basename(path)}: 样本 {t:,} 条，fid==-1 记录 {nneg1:,} 条，异常 {len(bad)} 条")

    print("\n====== 汇总 ======")
    print(f"总样本数：{grand_total:,}")
    print(f"fid == -1 的记录总数：{grand_fid_neg1:,}")
    if all_bad:
        print(f"❌ 有 {len(all_bad)} 条 fid==-1 的记录能量不为 0（或缺失能量字段，容差 atol={args.atol}）")
        # 列出前若干条异常，防止刷屏
        preview = 50
        print(f"以下列出前 {min(preview, len(all_bad))} 条：")
        for (db, key, e) in all_bad[:preview]:
            print(f"  - {os.path.basename(db)} [key={key}]  energy={e}")
        if len(all_bad) > preview:
            print(f"... 其余 {len(all_bad) - preview} 条已省略")
    else:
        print("✅ 所有 fid==-1 的记录能量均为 0（在给定容差内）")

if __name__ == "__main__":
    main()


In [ ]:
from adsorbdiff.utils.utils import build_config
from types import SimpleNamespace
cfg = build_config(SimpleNamespace(mode='train', config_yml='configs/flow/painn_so3_flow.yml')
, override_args=[])
print("EFFECTIVE imports =", cfg.get('imports'))
print("EFFECTIVE trainer =", cfg.get('trainer'))

In [ ]:
from adsorbdiff.datasets import LmdbDataset
ds = LmdbDataset({"src": "0/final_struct_lmdb"})
for d in ds:
      print(d.atomic_numbers)


In [ ]:
import os

print(os.getcwd())  # 当前工作目录
print(os.listdir(""))  # 看看目录里到底有哪些文件夹

In [ ]:
import pickle

# 修改为你的文件路径
file_path = "oc20_dense_mappings/oc20dense_compute.pkl"

# 打开并加载 pkl 文件
with open(file_path, "rb") as f:
    data = pickle.load(f)

# 打印对象类型
print("Loaded object type:", type(data))

# 打印前几个内容（根据对象类型不同显示方式也不同）
if isinstance(data, dict):
    print("Keys:", len(data.keys()))
if isinstance(data, dict):
    print("Keys:", list(data.keys()))
elif isinstance(data, (list, tuple)):
    print("Length:", len(data))
    print("First 5 elements:", data[:5])
else:
    print("Content preview:", str(data)[:500])  # 打印前 500 个字符
print(data['0_1190_0'])
